<a href="https://colab.research.google.com/github/raomv/ApplicationsNLP/blob/P2/sesion_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data download

To download the necessary datasets for our analysis, we use the `wget` command to fetch files from specified URLs. These files include a news text file in English for morphological tagging, an audio file for speech recognition, a corpus of scientific articles for text classification, a corpus of mobile phone reviews, and a Twitter corpus with ironic and non-ironic messages.

In [ ]:
!wget https://www.dlsi.ua.es/~dtomas/apln/news.txt  # News file in English for morphological tagging
!wget https://www.dlsi.ua.es/~dtomas/apln/audio.mp3  # Audio for speech recognition
!wget https://www.dlsi.ua.es/~dtomas/apln/repec_s.csv  # Corpus of scientific articles for text classification
!wget https://www.dlsi.ua.es/~dtomas/apln/cell_phones.csv  # Corpus of mobile phone reviews
!wget https://www.dlsi.ua.es/~dtomas/apln/irony.csv  # Twitter corpus with ironic and non-ironic messages

## Initial configuration for the notebook

In this section, we set up the initial configuration for the notebook. This includes installing required libraries, importing essential modules, and loading various models. We start by installing libraries such as Whisper and FAISS. We then import a wide range of libraries for data manipulation, visualization, natural language processing, and machine learning. Additionally, we download and load language models for SpaCy and NLTK, ensuring that all tools and resources are ready for subsequent analysis and tasks.

In [ ]:
# Install Gensim
!pip install gensim

# Install Whisper library
!pip install git+https://github.com/openai/whisper.git

# Install FAISS library
!pip install faiss-cpu  # There is a version in Conda for GPU

In [ ]:
import csv  # Handle CSV files
import faiss
from faiss import write_index
from gensim.corpora import Dictionary  # Mapping between words and IDs
from gensim.models import LdaModel  # Load LDA model
from gensim.parsing.preprocessing import STOPWORDS  # List of stopwords
from gensim.utils import simple_preprocess  # Convert a document into a list of tokens
import matplotlib.pyplot as plt  # Visualization
import nltk  # Natural Language Toolkit
from nltk.stem import WordNetLemmatizer  # Lemmatization
import numpy as np  # Numerical operations
import pandas as pd  # Table manipulation
import seaborn as sns  # Visualization
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF weighting
from sklearn.metrics import accuracy_score  # Classifier accuracy
from sklearn.metrics import confusion_matrix  # Confusion matrix
from sklearn.model_selection import cross_val_score  # Cross-validation
from sklearn.model_selection import train_test_split  # Train/test split
from sklearn.naive_bayes import MultinomialNB  # Naïve Bayes
from sklearn.neighbors import KNeighborsClassifier  # k-NN
from sklearn.neural_network import MLPClassifier  # Multi-layer perceptron
from sklearn.svm import SVC  # SVM
from sklearn.tree import DecisionTreeClassifier  # Decision tree
import spacy  # POS-tagger and parsing
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
from transformers import pipeline  # Pipelines for inference
from transformers import BertModel, BertTokenizer  # BERT for word embeddings
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AdamW  # Use pre-trained models
import whisper  # Use OpenAI's Whisper, which is excellent, though not available on Huggingface
from wordcloud import WordCloud  # Create word clouds

# Download the English model for SpaCy
spacy.cli.download('en_core_web_sm')

# Load the English model for SpaCy
nlp = spacy.load('en_core_web_sm')

# Download WordNet data for lemmatization
nltk.download('wordnet')

# Morphological Tagging and Syntactic Analysis

Morphological tagging, also known as part-of-speech (POS) tagging, is the process of assigning a part of speech to each word in a text, such as noun, verb, adjective, etc. It involves analyzing the structure of words and their components, including prefixes, suffixes, and roots, to determine their grammatical category. Syntactic analysis, or parsing, involves analyzing the structure of sentences to understand the relationships between words and how they combine to form phrases and sentences. This includes identifying the grammatical roles of words (such as subject, object, and predicate) and constructing a syntactic tree that represents the hierarchical structure of the sentence.

## Recipe 1

In this recipe, we will perform morphological tagging and syntactic analysis using the SpaCy library. We will load the English language model, process a sample text, and extract sentences and morphological information. We will also visualize the frequency of different parts of speech, explore noun phrases, and navigate the dependency tree. Finally, we will highlight named entities in the text.

In [ ]:
# Process the example text with SpaCy
text = 'Today is Wednesday, March 13, 2024. It is approximately 4:30 PM and I am in the Natural Language Processing Applications (APLN) class, here at the University of Alicante, in Spain. The professor is David. He tries to make the class interesting, but sometimes he fails.'
document = nlp(text)

In [ ]:
# Extract the list of sentences from the text to see how they look
list(document.sents)

In [ ]:
# Extract morphological information (POS-tagging) for each word in the text
for token in document:
  print('Word: ' + token.text)
  print('Lemma: ' + token.lemma_)
  print('POS: ' + token.pos_)
  print('Fine POS: ' + token.tag_)
  print('---')

In [ ]:
# You can use the "explain" method if you want to know more about a tag
spacy.explain('CD')

In [ ]:
# Create a DataFrame based on the content for further analysis
df = pd.DataFrame(data=[[token.text, token.lemma_, token.pos_, token.tag_] for token in document], columns=['Word', 'Lemma', 'POS', 'Fine POS'])
df

In [ ]:
# Basic statistics of the columns
df.describe()

In [ ]:
# How many verbs are in the text?
# You can replace 'VERB' with any other label (e.g., 'PUNCT')
(df['POS'] == 'VERB').sum()

In [ ]:
# We can create some interesting visualizations
# Here is a bar chart with the frequency of each tag
plt.figure(figsize=(14,7))
sns.countplot(x='POS', data=df)
plt.xticks(rotation=-45)  # Rotate labels to avoid overlap
plt.show()

## Recipe 2

In this recipe, we will explore noun phrases and the dependency tree of a given text using the SpaCy library. We will retrieve noun phrases from the text and visualize the dependency tree to understand the syntactic structure.

In [ ]:
# Retrieve noun phrases from the text
for chunk in document.noun_chunks:
  print('Noun phrase: ' + chunk.text)

In [ ]:
# Use "displacy" to display the dependency tree
spacy.displacy.render(document, style='dep', options={'compact': True}, jupyter=True)

In [ ]:
# Navigate the dependency tree
#   - "head" and "child" describe words connected by an arc in the dependency tree
#   - "dep" is the syntactic relation connecting "child" to "head"
for token in document:
  print(token.text, token.dep_, token.head.text, token.head.pos_, [child for child in token.children])

# Named Entity Recognition

Named Entity Recognition (NER) is a subtask of information extraction that seeks to locate and classify named entities mentioned in unstructured text into predefined categories such as the names of persons, organizations, locations, expressions of times, quantities, monetary values, percentages, etc. NER is used in various applications such as content classification, question answering, and information retrieval. It helps in understanding the context and meaning of the text by identifying and categorizing key elements.

## Recipe 1

In this recipe, we will use the Huggingface library to perform NER on a sample text. We will utilize a pre-trained model to identify and classify entities such as names of people, organizations, and locations.

In [ ]:
# Use the Huggingface library for an NER model
# If you want to use another model, modify: pipeline("ner", model="...")
text = 'My name is David and I work at the University of Alicante, here in Spain.'

ner = pipeline('ner', grouped_entities=True)#model =
ner(text)

## Recipe 2

In this recipe, we will use the SpaCy library to perform NER on a sample text. We will load the English language model, process the text, and extract entities such as names of people, organizations, and locations. We will then visualize these entities using SpaCy's `displacy` tool to highlight the identified entities within the text.

In [ ]:
# Use SpaCy for NER as well
# The complete list of recognized entities can be seen by executing "nlp.get_pipe('ner').labels"
document = nlp(text)

for ent in document.ents:
  print('Text: ' + ent.text)
  print('Start char: ' + str(ent.start_char))
  print('End char: ' + str(ent.end_char))
  print('Type: ' + ent.label_)
  print('---')

In [ ]:
# Highlight entities and their labels in the text using "displacy"
spacy.displacy.render(document, style='ent')

## Exercise

Highlight named entities in the `news.txt` file using "displacy".

In [ ]:
import spacy
from spacy import displacy

#SpaCy
nlp = spacy.load("en_core_web_sm")

with open("news.txt", "r", encoding="utf-8") as file:
    news_text = file.read()

doc = nlp(news_text)

#Display
displacy.render(doc, style="ent", jupyter=True)


# Topic Modeling

Topic modeling is a type of statistical model used to discover abstract topics within a collection of documents. It helps in identifying patterns and themes by clustering similar words together, making it easier to understand the underlying structure of the text data. Common algorithms for topic modeling include Latent Dirichlet Allocation (LDA) and Non-Negative Matrix Factorization (NMF). These models are widely used in various applications such as document classification, recommendation systems, and text summarization.

## Recipe 1

In this recipe, we will preprocess a small set of documents, create a dictionary and a corpus, and build an LDA model using the Gensim library. We will then analyze the results to understand the dominant topics in the dataset.

In [ ]:
# Create a small set of "documents"
documents = [
  'Human machine interface for lab abc computer applications',
  'A survey of user opinion of computer system response time',
  'The EPS user interface management system',
  'System and human system engineering testing of EPS',
  'Relation of user perceived response time to error measurement',
  'The generation of random binary unordered trees',
  'The intersection graph of paths in trees',
  'Graph minors IV Widths of trees and well quasi ordering',
  'Graph minors A survey'
]

In [ ]:
# Preprocess: tokenize the text, remove stopwords, and lemmatize
def preprocess(text):
  result = []
  for token in simple_preprocess(text):
    if token not in STOPWORDS and len(token) > 3:
      result.append(WordNetLemmatizer().lemmatize(token))
  return result

# Preprocess the documents
processed_docs = [preprocess(doc) for doc in documents]
processed_docs

In [ ]:
# Gensim requires a dictionary and a corpus:
# - The dictionary maps each word to a unique identifier
# - The corpus is a list of vectors where each vector represents a document
dictionary = Dictionary(processed_docs)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
corpus

In [ ]:
# Build the LDA model using the corpus and dictionary
lda_model = LdaModel(corpus=corpus, # The dataset
                     id2word=dictionary, # Dictionary mapping each word to a unique ID
                     num_topics=5, # Number of topics to identify
                     random_state=666, # Set a seed for reproducibility
                     update_every=1, # Update model parameters every X documents
                     passes=15, # Number of passes over the complete corpus
                     alpha='auto', # Controls the number of topics per document
                     per_word_topics=True) # Also calculates the most probable topics for each word

In [ ]:
# Display the most relevant words for each topic along with their weight
topics = lda_model.print_topics(num_words=5)
for topic in topics:
  print(topic)

In [ ]:
# Assign topics to each document in the corpus
for document_num, doc in enumerate(corpus):
  # Get topic distribution for the document
  doc_topics, word_topics, phi_values = lda_model.get_document_topics(doc, per_word_topics=True)

  # Sort by the second element of the tuple (topic probability) and get the highest
  dominant_topic = sorted(doc_topics, key=lambda x: x[1], reverse=True)[0]
  topic_num, prop_topic = dominant_topic
  # Display document number, dominant topic, and associated words
  print(f"Document {document_num}: Main topic {topic_num}, weight {prop_topic:.4f}, words {lda_model.show_topic(topic_num)}")

# Information Extraction

Information extraction (IE) is the process of automatically extracting structured information from unstructured or semi-structured text. This includes identifying and categorizing key elements such as names of people, organizations, locations, dates, and other entities, as well as relationships between them. IE is used in various applications such as data mining, knowledge discovery, and natural language processing to transform raw text into meaningful and actionable data. Techniques used in IE include NER, relation extraction, and event extraction.

## Recipe 1

In this recipe, we will extract specific information from a contract text using the SpaCy library. We will process the text to identify and extract entities such as the seller, buyer, company, and location. This involves using named entity recognition (NER) to detect relevant entities and assigning them to the appropriate variables. Finally, we will display the extracted information and visualize the entities within the text using SpaCy's `displacy` tool.

In [ ]:
# Create an example contract text
contract_text = '''
This purchase agreement is entered into between Juan Pérez, from the company Autos Locos S.A., hereinafter referred to as "The Seller", and Ana López, hereinafter referred to as "The Buyer", who agree as follows:

The Seller agrees to sell and The Buyer agrees to buy the vehicle described below, under the terms and conditions established in this contract.

The contract is signed in the city of Madrid (Spain), March 27, 2020.
'''

In [ ]:
# Process the text with SpaCy
document = nlp(contract_text)

# Initialize variables to store extracted information
seller = buyer = company = location = date = None

# Iterate over detected entities
# Adjust the logic to assign detected entities:
#   - Assign the first found name as the seller
#   - Assign the next found name as the buyer
#   - Assign the first detected company as the company in the contract
#   - Assign the first detected location as the signing location
for ent in document.ents:
  if ent.label_ == 'PERSON' and seller is None:
    seller = ent.text
  elif ent.label_ == 'PERSON' and seller is not None and buyer is None:
    buyer = ent.text
  elif ent.label_ == 'ORG' and company is None:
    company = ent.text
  elif ent.label_ == 'GPE' and location is None:
    location = ent.text
  elif ent.label_ == 'DATE' and date is None:
    date = ent.text

# Verify if expected values were found and display results
print(f"Seller: {seller if seller else 'Not identified'}")
print(f"Buyer: {buyer if buyer else 'Not identified'}")
print(f"Company: {company if company else 'Not identified'}")
print(f"Signing Location: {location if location else 'Not identified'}")
print(f"Date: {date if date else 'Not identified'}")

# Speech Recognition

Speech recognition is the process of converting spoken language into written text. It involves capturing audio signals, processing them to identify linguistic content, and transcribing the spoken words into text. This technology is used in various applications such as virtual assistants, transcription services, voice-controlled devices, and accessibility tools for individuals with disabilities. Speech recognition systems typically use machine learning models trained on large datasets of spoken language to accurately recognize and transcribe speech.

## Recipe 1

In this recipe, we will use the Whisper library to perform speech recognition on an audio file. We will load a pre-trained model, transcribe the audio file, and display the transcribed text. This approach allows us to convert spoken language into written text, enabling further analysis and processing of the transcribed content.

In [ ]:
# Load the model
model = whisper.load_model('base')

# Transcribe the audio file
result = model.transcribe('audio.mp3')

# Display the transcribed text
result['text']

# Question Answering

Question answering (QA) is a task in NLP that involves automatically answering questions posed by humans in natural language. It typically involves understanding the context of a given text and extracting relevant information to provide accurate answers. QA systems can be categorized into different types, such as extractive QA, where the answer is a span of text from the context, and abstractive QA, where the answer is generated based on the understanding of the context. QA models are widely used in applications like virtual assistants, customer support, and information retrieval.

## Recipe 1

In this recipe, we will use the Huggingface library to perform question answering on a sample text. We will create a pipeline for question answering, provide a question and a context where the answer can be found, and retrieve the answer from the context.

In [ ]:
# Create the pipeline
question_answerer = pipeline('question-answering')

# Provide the question and context where the answer is found
question_answerer(
  question = 'Where does Manolo work?',
  context = 'Julia is an engineer. She loves sushi. Her husband\'s name is Manolo, but they don\'t live together. She lives in Japan and he lives in New Zealand.'
)

## Exercise

Find a Spanish model that performs well on this task.

In [ ]:
question_answerer = pipeline("question-answering", model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

a = question_answerer(
    question="¿Dónde trabaja Manolo?",
    context="Julia es ingeniera. Le encanta el sushi. El nombre de su esposo es Manolo, pero no viven juntos. Ella vive en Japón y él vive en Nueva Zelanda."
)

print(a)

b = question_answerer(
    question="¿Qué enciendes primero?",
    context="Estás en un laberinto completamente a oscuras y tienes una vela, una antorcha, un puñado de paja y una sola cerilla"
)

print(b)


# Text Classification

Text classification is a supervised learning technique used to categorize text into predefined classes or labels. It involves training a model on a labeled dataset, where each text sample is associated with a specific category. The model learns to identify patterns and features in the text that correspond to each category. Once trained, the model can predict the category of new, unseen text samples. Text classification is widely used in various applications such as spam detection, sentiment analysis, topic labeling, and document categorization. Techniques for text classification include traditional machine learning algorithms like Naïve Bayes, SVM, and Decision Trees, as well as advanced deep learning models like Transformers.

## Recipe 1

In this recipe, we will explore zero-shot classification using the Huggingface library. Zero-shot classification allows us to classify text into categories that the model has not been explicitly trained on. We will use a pre-trained model to classify a sample text into one of the specified candidate labels. This approach is useful when we need to categorize text into new or custom labels without additional training.

In [ ]:
# Zero-shot classification implies that the model has not been trained for these specific classes
classifier = pipeline('zero-shot-classification')
classifier(
  'This is a course about the Transformers library',
  candidate_labels=['education', 'politics', 'business'],
)

## Recipe 2

In this recipe, we will use classic machine learning algorithms to classify scientific articles based on their titles. We will preprocess the text data, create feature vectors using TF-IDF, and train classifiers such as Decision Trees, k-NN, Neural Networks, Naïve Bayes, and SVM. We will evaluate the performance of these classifiers using train/test split and cross-validation methods.

In [ ]:
# Let's use some "classic" algorithms to train our classifier
# Load the model while disabling some functionalities we won't use to speed it up
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'entity_linker', 'entity_ruler'])

# Show the first few lines of the "repec_s.csv" file to see what it looks like
!head repec_s.csv

In [ ]:
# Load data into a Pandas DataFrame
data = pd.read_csv('repec_s.csv')
data

In [ ]:
# We will classify based on the "abstract" (the summary of the article)
corpus = data['title']  # Store the abstract
y = data['jel']  # Store the JEL category

In [ ]:
# Show a bar chart with the number of instances per class
plt.figure(figsize=(10,8))
sns.countplot(x=y)
plt.show()

# F: International Economics
# I: Health, Education, and Welfare
# R: Urban, Rural, Regional, Real Estate, and Transportation Economics
# M: Business Administration and Business Economics | Marketing | Accounting | Personnel Economics

In [ ]:
# Preprocessing: remove punctuation, stopwords, and convert to lowercase
def normalize(text):
  document = nlp(text)  # Process text with SpaCy
  document = [token for token in document if not token.is_punct]  # Remove punctuation
  document = [token for token in document if not token.is_stop]  # Remove stopwords
  document = [token.lower_ for token in document]  # Convert to lowercase
  return ' '.join(document)

corpus_normalized = corpus.map(normalize)
corpus_normalized

In [ ]:
# Create a function "classify" that performs training and testing
# Input parameters:
#   - corpus: dataset containing training and test data
#   - model_name: name of the algorithm to use ("DT", "KNN", "MLP", "NB" or "SVM")
#   - evaluation_type: evaluation type, either train/test split ("split") or cross-validation ("cv")
# The function returns the trained model and the "vectorizer"
# We need both if we want to predict new instances

def classify(corpus, model_name, evaluation_type):
  vectorizer = TfidfVectorizer(ngram_range=(1,2))
  X = vectorizer.fit_transform(corpus)

  if model_name == 'DT':
    model = DecisionTreeClassifier()  # Decision tree
  elif model_name == 'KNN':
    model = KNeighborsClassifier()  # k-NN
  elif model_name == 'MLP':
    model = MLPClassifier()  # Neural network
  elif model_name == 'NB':
    model = MultinomialNB()  # Naïve Bayes
  else:
    model = SVC(kernel='linear')  # SVM

  # If the user chooses train/test split
  if evaluation_type == 'split':
    # Split into 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

    # Train the model
    model.fit(X_train, y_train)

    # Predict on the test set
    predictions = model.predict(X_test)

    # Compute model accuracy
    print('Accuracy: {:.2%}\n'.format(accuracy_score(predictions, y_test)))
    print('Confusion matrix:')

    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, predictions), annot=True, linewidth=3)
    plt.yticks(rotation=0)
    plt.show()
  # If the user chooses cross-validation
  elif evaluation_type == 'cv':
    scores = cross_val_score(model, X, y, cv=5)  # 5-fold
    print('%0.2f accuracy with a standard deviation of %0.2f' % (scores.mean(), scores.std()))
  else:
    print('Unknown evaluation type')

  return model, vectorizer

In [ ]:
# Possible values for model_name: "DT", "KNN", "MLP", "NB", "SVM"
# Possible values for evaluation_type: "split", "cv"
# Returns the trained model and the vectorizer for future predictions

model_repec, vectorizer_repec = classify(corpus_normalized, 'SVM', 'split')

In [ ]:
# If you want to know which class corresponds to each number...
model_repec.classes_

In [ ]:
# Predict the class for a new (previously unseen) sample
new_input = ['This study examines the relationship between global banana prices and climate change, analyzing decades of data through statistical and machine learning methods. We find that climate variables, such as temperature rises and extreme weather, significantly impact banana cultivation and pricing. Our results highlight the importance of incorporating climate resilience in agricultural policies, particularly for banana-dependent economies, to mitigate global warming\'s effects on food security and economic stability.']

new_input = vectorizer_repec.transform(new_input)  # Transform the new sample following the same procedure used when creating the model
label = model_repec.predict(new_input)  # Predict the class (F, I, R, or M)

if label == 'F':
  print('International Economics')
elif label == 'I':
  print('Health, Education, and Welfare')
elif label == 'R':
  print('Urban, Rural, Regional, Real Estate, and Transportation Economics')
elif label == 'M':
  print('Business Administration and Business Economics | Marketing | Accounting | Personnel Economics')
else:
  print('Unknown class')  # This should never happen

## Recipe 3

In this recipe, we will use BERT embeddings to represent sentences and classify scientific articles based on their titles. We will preprocess the text data, create embeddings using a pre-trained BERT model, and train classifiers such as Decision Trees, k-NN, Neural Networks, Naïve Bayes, and SVM. We will evaluate the performance of these classifiers using train/test split and cross-validation methods.

In [ ]:
# Repeat the experiment above but using BERT to represent sentences
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Check if GPU is available
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # Tokenizer to extract words
model = BertModel.from_pretrained('bert-base-uncased').to(device)  # Basic BERT model

# Function that receives text and returns its embedding using BERT
def encode(text):
  input_ids = tokenizer.encode(text, max_length=512, truncation=True, return_tensors='pt').to(device)  # Limit to 512 tokens (BERT's max length)
  with torch.no_grad():  # Disable gradient calculation during inference
    features = model(input_ids)[0][0][0].tolist()  # The first token ('[CLS]') encodes the sentence embedding
  return features

# Similar to "classify" but receives embeddings instead of raw text
def classify_BERT(X, model_name, evaluation_type):
  if model_name == 'DT':
    model = DecisionTreeClassifier()  # Decision tree
  elif model_name == 'KNN':
    model = KNeighborsClassifier()  # k-NN
  elif model_name == 'MLP':
    model = MLPClassifier()  # Neural network
  elif model_name == 'NB':
    model = MultinomialNB()  # Naïve Bayes
  else:
    model = SVC(kernel='linear')  # SVM

  # Train/test split
  if evaluation_type == 'split':
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

    # Train the model
    model.fit(X_train, y_train)

    # Predict on the test set
    predictions = model.predict(X_test)

    # Compute accuracy
    print('Accuracy: {:.2%}\n'.format(accuracy_score(predictions, y_test)))
    print('Confusion matrix:')

    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, predictions), annot=True, linewidth=3)
    plt.yticks(rotation=0)
    plt.show()
  # Cross-validation
  elif evaluation_type == 'cv':
    scores = cross_val_score(model, X, y, cv=5)  # 5-fold
    print("%0.2f accuracy with a standard deviation of %0.2f" % (scores.mean(), scores.std()))
  else:
    print('Unknown evaluation type')

  return model

In [ ]:
# Create the corpus with all article abstracts
# You will need a GPU to get the corpus processed in a decent time
# Each abstract is encoded and stored in X
y = data['jel']  # Store the JEL category
X = [encode(abstract) for abstract in corpus_normalized]

classify_BERT(X, 'SVM', 'split')

## Exercise

Try classifying using the titles instead of the abstracts.

In [ ]:
corpus = data['title']  # Store the title
y = data['jel']  # Store the JEL category

# Preprocessing remains the same
corpus_normalized = corpus.map(normalize)

# Create the corpus with all article titles
X = np.array([encode(title) for title in corpus_normalized])
X = X.reshape(X.shape[0], -1) # Reshape to 2D: (num_samples, num_features)

classify_BERT(X, 'SVM', 'split')

Try using different n-gram sizes.

Hint: `vectorizer = TfidfVectorizer(ngram_range=(1,2))  # Uses 1-grams and 2-grams`

In [ ]:
'''
Si ngram_range=(1,1): Solo se considerarán palabras individuales (unigramas) como características.
Por ejemplo, "el", "gato", "come".
Si ngram_range=(1,2): Se considerarán palabras individuales (unigramas) y pares de palabras
consecutivas (bigramas) como características. Por ejemplo, "el gato", "gato come", además de "el", "gato", "come".
Si ngram_range=(2,3): Se considerarán pares de palabras (bigramas) y tríos de palabras consecutivas
(trigramas) como características. Por ejemplo, "el gato come", "gato come pescado", además de "el gato", "gato come"
'''

def classify2(corpus, model_name, evaluation_type):
  vectorizer = TfidfVectorizer(ngram_range=(2,3))
  X = vectorizer.fit_transform(corpus)

  if model_name == 'DT':
    model = DecisionTreeClassifier()  # Decision tree
  elif model_name == 'KNN':
    model = KNeighborsClassifier()  # k-NN
  elif model_name == 'MLP':
    model = MLPClassifier()  # Neural network
  elif model_name == 'NB':
    model = MultinomialNB()  # Naïve Bayes
  else:
    model = SVC(kernel='linear')  # SVM

  # If the user chooses train/test split
  if evaluation_type == 'split':
    # Split into 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

    # Train the model
    model.fit(X_train, y_train)

    # Predict on the test set
    predictions = model.predict(X_test)

    # Compute model accuracy
    print('Accuracy: {:.2%}\n'.format(accuracy_score(predictions, y_test)))
    print('Confusion matrix:')

    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, predictions), annot=True, linewidth=3)
    plt.yticks(rotation=0)
    plt.show()
  # If the user chooses cross-validation
  elif evaluation_type == 'cv':
    scores = cross_val_score(model, X, y, cv=5)  # 5-fold
    print('%0.2f accuracy with a standard deviation of %0.2f' % (scores.mean(), scores.std()))
  else:
    print('Unknown evaluation type')

  return model, vectorizer


model_repec, vectorizer_repec = classify2(corpus_normalized, 'SVM', 'split')

What about using a different word embedding model?

In [ ]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
# Repeat the experiment above but using BERT to represent sentences
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # Tokenizer to extract words
model = SentenceTransformer('all-mpnet-base-v2').to(device)

# Function that receives text and returns its embedding using BERT
def encode(text):
  embeddings = model.encode(text)
  return embeddings.tolist()

# Similar to "classify" but receives embeddings instead of raw text
def classify_BERT2(X, model_name, evaluation_type):
  if model_name == 'DT':
    model = DecisionTreeClassifier()  # Decision tree
  elif model_name == 'KNN':
    model = KNeighborsClassifier()  # k-NN
  elif model_name == 'MLP':
    model = MLPClassifier()  # Neural network
  elif model_name == 'NB':
    model = MultinomialNB()  # Naïve Bayes
  else:
    model = SVC(kernel='linear')  # SVM

  # Train/test split
  if evaluation_type == 'split':
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

    # Train the model
    model.fit(X_train, y_train)

    # Predict on the test set
    predictions = model.predict(X_test)

    # Compute accuracy
    print('Accuracy: {:.2%}\n'.format(accuracy_score(predictions, y_test)))
    print('Confusion matrix:')

    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, predictions), annot=True, linewidth=3)
    plt.yticks(rotation=0)
    plt.show()
  # Cross-validation
  elif evaluation_type == 'cv':
    scores = cross_val_score(model, X, y, cv=5)  # 5-fold
    print("%0.2f accuracy with a standard deviation of %0.2f" % (scores.mean(), scores.std()))
  else:
    print('Unknown evaluation type')

  return model

In [ ]:
y = data['jel']  # Store the JEL category
X = [encode(abstract) for abstract in corpus_normalized]

classify_BERT2(X, 'SVM', 'split')

# Sentiment Analysis

Sentiment analysis is an NLP technique used to determine the emotional tone or sentiment expressed in a piece of text. It involves classifying text into categories such as positive, negative, or neutral based on the sentiments conveyed. Sentiment analysis is widely used in various applications, including social media monitoring, customer feedback analysis, and market research, to understand public opinion and sentiment trends. Techniques for sentiment analysis range from simple rule-based approaches to advanced machine learning models, including deep learning and transformer-based models.

## Recipe 1

In this recipe, we will use the Huggingface library to perform sentiment analysis on a sample text. We will create a pipeline for sentiment analysis, provide a text input, and retrieve the sentiment classification (positive, negative, or neutral) from the model.

In [ ]:
# Using the Transformers pipeline
classifier = pipeline('sentiment-analysis')

classifier('Today is Friday.')

In [ ]:
# Now without a pipeline, using a pre-trained model
# Texts to analyze
raw_inputs = [
    'I have been waiting for this course my whole life.',
    'I hate this so much!',
]

# Model to use
checkpoint = 'distilbert-base-uncased-finetuned-sst-2-english'

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# Tokenization and classification
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
outputs = model(**inputs)

# Convert logits to probabilities for each label
import torch
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(model.config.id2label)  # Show which label corresponds to each prediction
print(predictions)

## Recipe 2

In this recipe, we will use a pre-trained sentiment analysis model to classify the sentiment of mobile phone reviews. We will load the dataset, preprocess the text, and use the model to predict whether each review is positive or negative. Finally, we will visualize the results and create a word cloud to highlight the most frequent words in the reviews.

In [ ]:
# Repeat the task using the text classifier
# We'll use a corpus of mobile phone reviews. Let's check the first few lines.
!head cell_phones.csv

In [ ]:
# Use Pandas for better corpus handling
data = pd.read_csv('cell_phones.csv')
data

In [ ]:
# Extract comments and labels
corpus = data['content']  # Store comments
y = data['opinion']  # Store labels ("POS" and "NEG")

In [ ]:
# Show a bar chart with class distribution
plt.figure(figsize=(5,4))
sns.countplot(x=y)
plt.show()

In [ ]:
# Preprocessing. Reuse the "normalize" function defined above
corpus_normalized = corpus.map(normalize)
corpus_normalized

In [ ]:
# Use the "classify" function as before
# Possible values for model_name: "DT", "KNN", "MLP", "NB", "SVM"
# Possible values for evaluation_type: "split", "cv"
model_phones, vectorizer_phones = classify(corpus_normalized, 'SVM', 'split')

In [ ]:
# Prediction
new_input = ['I love this phone!!']
new_input = vectorizer_phones.transform(new_input)
label = model_phones.predict(new_input)  # Predict label ("POS" or "NEG")

if label == 'POS':
  print('Positive opinion')
elif label == 'NEG':
  print('Negative opinion')
else:
  print('Unknown class')

## Recipe 3

In this recipe, we will create a word cloud to visualize the most frequent words in mobile phone reviews. We will preprocess the text data by removing stopwords, punctuation, and converting the text to lowercase. Then, we will generate a word cloud using the WordCloud library and display it using Matplotlib. This visualization helps in understanding the common themes and sentiments expressed in the reviews.

In [ ]:
# Create a word cloud with content from 'cell_phones.csv'
list_rows = []  # Store all rows from the file (content only)
with open('cell_phones.csv') as file:
  csv_reader = csv.DictReader(file, delimiter=',', quotechar='"')

  for row in csv_reader:
    list_rows.append(row['content'])

corpus = nlp(' '.join(list_rows))  # Concatenate all sentences into a single string
tokens = [w.lower_ for w in corpus if (not w.is_space and not w.is_punct and not w.is_stop)]  # Create corpus from words in comments
corpus = ' '.join(tokens)  # Merge all words into a single string

wordcloud = WordCloud(width=800, height=800, background_color='white', min_font_size=10).generate(corpus)  # Create word cloud

plt.figure(figsize=(8, 8), facecolor=None)  # Display word cloud as an image
plt.imshow(wordcloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

# Irony Detection

Irony detection is the task of identifying ironic statements in text. Irony is a form of expression where the intended meaning is opposite to the literal meaning, often used for humorous or emphatic effect. Detecting irony is challenging because it relies on contextual and cultural knowledge, as well as an understanding of subtle cues in language. Techniques for irony detection include machine learning models trained on annotated datasets, natural language processing methods to analyze linguistic features, and deep learning approaches such as BERT to capture contextual information. Applications of irony detection include sentiment analysis, social media monitoring, and improving the accuracy of natural language understanding systems.

## Recipe 1

In this recipe, we will explore irony detection using a pre-trained BERT model. We will load the irony dataset, preprocess the text, and fine-tune a classification model to detect ironic and non-ironic messages. We will then evaluate the model's performance and analyze the results.

In [ ]:
# Let's check what the irony dataset looks like
!head irony.csv

In [ ]:
# Load data into a DataFrame
data = pd.read_csv('irony.csv', sep='\t', index_col=0)
data.Label = data.Label.astype(int)  # Convert labels to integers to avoid PyTorch issues

labels = data.Label.tolist()  # List of labels (0: non-ironic, 1: ironic)
texts = data.Text.tolist()  # List of messages

In [ ]:
# Tokenize texts
checkpoint = 'distilbert/distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512)

In [ ]:
# Fine-tune a classification model for irony detection
labels = torch.tensor(labels)

# Create a PyTorch dataset
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)

# Split dataset into training and testing
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders for training and testing
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# Check if GPU is available and load the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model.to(device)  # Move model to the appropriate device
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training loop (basic, without validation)
model.train()
for epoch in range(4):  # Number of epochs
  for batch in train_loader:
    batch = [b.to(device) for b in batch]  # Move data to the appropriate device
    input_ids, attention_mask, labels = batch
    optimizer.zero_grad()
    outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
  print(f'Epoch {epoch+1}: Loss {loss.item()}')

In [ ]:
# Evaluate the model
def evaluate(model, test_loader):
  model.eval()
  predictions, true_labels = [], []

  with torch.no_grad():
    for batch in test_loader:
      batch = [b.to(device) for b in batch]  # Move data to the appropriate device
      input_ids, attention_mask, labels = batch
      outputs = model(input_ids, attention_mask=attention_mask)
      logits = outputs.logits
      predictions.extend(torch.argmax(logits, -1).tolist())
      true_labels.extend(labels.tolist())

  accuracy = accuracy_score(true_labels, predictions)
  print(f"Accuracy: {accuracy}")

evaluate(model, test_loader)

# Information Retrieval

Information retrieval (IR) is the process of obtaining relevant information from a large repository, such as a database or the internet, based on a user's query. It involves indexing, searching, and ranking documents to provide the most relevant results to the user.

## Recipe 1

In this recipe, we will explore how to use BERT embeddings and the FAISS library to perform information retrieval on a dataset of scientific article titles. We will preprocess the text data, create embeddings using a pre-trained BERT model, and build a FAISS index to enable efficient similarity search. Finally, we will demonstrate how to query the index to retrieve the most relevant documents based on a given query.

In [ ]:
# Move the model and data to the appropriate device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Function to obtain word embeddings with BERT
def get_bert_embeddings(documents):
  tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
  model = AutoModel.from_pretrained('bert-base-uncased').to(device)
  embeddings = []
  for doc in documents:
    inputs = tokenizer(doc, return_tensors='pt', truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
      outputs = model(**inputs)
    doc_embedding = outputs.last_hidden_state.mean(1).squeeze().cpu().numpy()
    embeddings.append(doc_embedding)
  return np.array(embeddings)

# Function to create the FAISS index
def create_faiss_index(embeddings):
  dimension = embeddings.shape[1]
  index = faiss.IndexFlatL2(dimension)
  index.add(embeddings)
  return index

# Read the CSV file
csv_file = 'repec_s.csv'
df = pd.read_csv(csv_file)
documents = df.title.tolist()  # The texts are in the "title" column

# Process the documents to obtain word embeddings
embeddings = get_bert_embeddings(documents)

# Create the FAISS index with the embeddings
index = create_faiss_index(embeddings)

# Save the index to disk if needed
# faiss.write_index(index, 'repec.index')

In [ ]:
# Function to obtain query embeddings
def get_query_embedding(query_text):
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    model = AutoModel.from_pretrained('bert-base-uncased').to(device)
    inputs = tokenizer(query_text, return_tensors='pt', truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    query_embedding = outputs.last_hidden_state.mean(1).squeeze().cpu().numpy()
    return query_embedding

# Example query
query_text = "prize of bananas"
query_embedding = get_query_embedding(query_text)
query_vector = np.expand_dims(query_embedding, axis=0)

# Perform search in the FAISS index
D, I = index.search(query_vector, k=5)  # Retrieve the 5 most similar documents

print("Query: ", query_text)
print("Most similar documents:")
for i, idx in enumerate(I[0], start=1):
    print(f"{i}: Document {idx} with a distance of {D[0][i-1]}")
    print("Document content:", documents[idx])

## Recipe 2

Based on [Build a semantic search engine](https://python.langchain.com/docs/tutorials/retrievers/) tutorial in [LangChain](https://www.langchain.com/).

Here we will build a search engine over a PDF document. This will allow us to retrieve passages in the PDF that are similar to an input query.

In [ ]:
# Install required libraries
!pip install langchain-community pypdf sentence-transformers langchain-chroma

### Documents and Document Loaders

LangChain implements a [Document](https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html) abstraction, which is intended to represent a unit of text and associated metadata. It has three attributes:

- `page_content`: a string representing the content;
- `metadata`: a dict containing arbitrary metadata;
- `id`: (optional) a string identifier for the document.

The `metadata` attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual `Document` object often represents a chunk of a larger document.

However, the LangChain ecosystem implements document loaders that integrate with hundreds of common sources. This makes it easy to incorporate data from these sources into your AI application.

### Loading documents

Let's load a PDF into a sequence of `Document` objects. There is a sample PDF in the LangChain repo [here](https://github.com/langchain-ai/langchain/tree/master/docs/docs/example_data) -- a 10-k filing for Nike from 2023. We can consult the LangChain documentation for available PDF document loaders. Let's select `PyPDFLoader`, which is fairly lightweight.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

`PyPDFLoader` loads one `Document` object per PDF page. For each, we can easily access:
* The string content of the page;
* Metadata containing the file name and page number.

In [ ]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

### Splitting

For both information retrieval and downstream question-answering purposes, a page may be too coarse a representation. Our goal in the end will be to retrieve `Document` objects that answer an input query, and further splitting our PDF will help ensure that the meanings of relevant portions of the document are not "washed out" by surrounding text.

We can use *text splitters* for this purpose. Here we will use a simple text splitter that partitions based on characters. We will split our documents into chunks of 1000 characters with 200 characters of overlap between chunks. The overlap helps mitigate the possibility of separating a statement from important context related to it. We use the `RecursiveCharacterTextSplitter`, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases.

We set `add_start_index=True` so that the character index where each split `Document` starts within the initial `Document` is preserved as metadata attribute `start_index`.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

### Embeddings

Vector search is a common way to store and search over unstructured data (such as unstructured text). The idea is to store numeric vectors that are associated with the text. Given a query, we can embed it as a vector of the same dimension and use vector similarity metrics (such as cosine similarity) to identify related text.

LangChain supports embeddings from dozens of providers. These models specify how text should be converted into a numeric vector.

In [ ]:
from langchain.embeddings import SentenceTransformerEmbeddings

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

### Vector stores

Armed with a model for generating text embeddings, we can next store them in a special data structure that supports efficient similarity search. LangChain [VectorStore](https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.base.VectorStore.html) objects contain methods for adding text and `Document` objects to the store, and querying them using various similarity metrics. They are often initialized with embedding models, which determine how text data is translated to numeric vectors.

LangChain includes a suite of *integrations* with different vector store technologies. Some vector stores are hosted by a provider (e.g., various cloud providers) and require specific credentials to use; some (such as *Postgres*) run in separate infrastructure that can be run locally or via a third-party; others can run in-memory for lightweight workloads. Let's select a vector store:

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(embedding_function=embeddings)

Having instantiated our vector store, we can now index the documents.

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

Note that most vector store implementations will allow you to connect to an existing vector store--  e.g., by providing a client, index name, or other information.

Once we've instantiated a `VectorStore` that contains documents, we can query it. [VectorStore](https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.base.VectorStore.html) includes methods for querying:
* Synchronously and asynchronously;
* By string query and by vector;
* With and without returning similarity scores;
* By similarity and [maximum marginal relevance](https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.base.VectorStore.html#langchain_core.vectorstores.base.VectorStore.max_marginal_relevance_search) (to balance similarity with query to diversity in retrieved results).

The methods will generally include a list of `Document` objects in their outputs.

### Usage

Embeddings typically represent text as a "dense" vector such that texts with similar meanings are geometrically close. This lets us retrieve relevant information just by passing in a question, without knowledge of any specific key-terms used in the document.

Return documents based on similarity to a string query:

In [ ]:
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

Async query (it tells your program to pause execution of the current code block until the asynchronous operation completes and returns a result):

In [ ]:
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

Return similarity scores:

In [ ]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Return documents based on similarity to an embedded query:

In [ ]:
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])